<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICS40125 - Laboratorio N°08

**Objetivo**: Aplicar técnicas de **machine learning no supervisado** para explorar, procesar y analizar conjuntos de datos con variables numéricas y categóricas.

> **Nota**: Puede ayudarse de algún asistente virtual como **ChatGPT, Gemini** u otros, así como del autocompletado de **Google Colab**, para avanzar en este laboratorio debido a su extensión.


## Clustering


<img src="https://www.svgrepo.com/show/253022/car.svg" width = "300" align="center"/>



El conjunto de datos **`vehiculos_procesado_con_grupos.csv`** recopila información sobre diversas características relevantes de distintos vehículos. El propósito de este ejercicio es **clasificar los vehículos en diferentes categorías**, utilizando como base las variables descritas en la tabla de atributos.

El análisis presenta un desafío adicional debido a la **naturaleza mixta de los datos**: se incluyen tanto variables **numéricas** (ej. dimensiones, consumo, emisiones) como **categóricas** (ej. tipo de tracción, tipo de combustible), lo que requiere aplicar técnicas de preprocesamiento adecuadas antes de entrenar los modelos.

Como primer paso, procederemos a **cargar y explorar el conjunto de datos**, con el fin de familiarizarnos con su estructura y las características que servirán como base para la posterior clasificación.




**Descripción de los Datos:**

| **Nombre de la Columna**   | **Descripción**                                                                                                                                   |
|----------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------|
| **year**                   | El año en que el vehículo fue fabricado.                                                                                                          |
| **desplazamiento**          | La capacidad volumétrica del motor en litros. Indica la cantidad de aire y combustible que puede desplazar el motor durante una revolución.       |
| **cilindros**               | El número de cilindros que tiene el motor. Los cilindros son las cámaras donde ocurre la combustión interna en los motores de los vehículos.       |
| **co2**                     | Emisiones de dióxido de carbono del vehículo, medido en gramos por kilómetro. Es una medida de las emisiones de gases de efecto invernadero.       |
| **clase_tipo**              | La clase o tipo de vehículo, como vehículos especiales, deportivos, etc.                                                                         |
| **traccion_tipo**           | Tipo de tracción del vehículo, ya sea tracción en dos ruedas, en cuatro ruedas o en todas las ruedas.                                             |
| **transmision_tipo**        | Tipo de transmisión del vehículo, como automática, manual, entre otros.                                                                          |
| **combustible_tipo**        | Tipo de combustible que utiliza el vehículo, como gasolina, diésel, eléctrico, híbrido, etc.                                                     |
| **tamano_motor_tipo**       | Clasificación del tamaño del motor (por ejemplo, pequeño, mediano o grande), que generalmente se basa en la capacidad de desplazamiento.           |
| **consumo_tipo**            | Clasificación del nivel de consumo de combustible del vehículo, indicando si es alto, bajo, o muy alto.                                           |
| **co2_tipo**                | Clasificación de las emisiones de CO2 del vehículo, indicando si es alto, bajo, o muy alto.                                                       |
| **consumo_litros_milla**    | El consumo de combustible del vehículo, medido en litros por milla. Indica la eficiencia del vehículo en términos de consumo de combustible.        |



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.cluster import KMeans


%matplotlib inline

sns.set_palette("deep", desat=.6)
sns.set(rc={'figure.figsize':(11.7,8.27)})

In [ ]:
# cargar datos
df = pd.read_csv("https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/vehiculos_procesado_con_grupos.csv", sep=",")\
       .drop(
            ["fabricante",
             "modelo",
             "transmision",
             "traccion",
             "clase",
             "combustible",
             "consumo"],

          axis=1)

df.head()

,year,desplazamiento,cilindros,co2,clase_tipo,traccion_tipo,transmision_tipo,combustible_tipo,tamano_motor_tipo,consumo_tipo,co2_tipo,consumo_litros_milla
0,1984,2.5,4.0,522.764706,Vehículos Especiales,dos,Automatica,Normal,pequeño,alto,alto,0.222671
1,1984,4.2,6.0,683.615385,Vehículos Especiales,dos,Automatica,Normal,grande,muy alto,muy alto,0.291185
2,1985,2.5,4.0,555.437500,Vehículos Especiales,dos,Automatica,Normal,pequeño,alto,alto,0.236588
3,1985,4.2,6.0,683.615385,Vehículos Especiales,dos,Automatica,Normal,grande,muy alto,muy alto,0.291185
4,1987,3.8,6.0,555.437500,Coches Medianos,dos,Automatica,Premium,grande,alto,alto,0.236588


En este caso, no solo se tienen datos numéricos, sino que también categóricos. Además, tenemos problemas de datos **vacíos (Nan)**. Así que para resolver este problema, seguiremos varios pasos:

### 1.- Normalizar datos

- Cree un conjunto de datos con las variables numéricas, además, para cada dato vacía, rellene con el promedio asociado a esa columna. Finalmente, normalize los datos mediante el procesamiento **MinMaxScaler** de **sklearn**.
- Cree un conjunto de datos con las variables categóricas , además, transforme de variables categoricas a numericas ocupando el comando **get_dummies** de pandas ([referencia](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.get_dummies.html)). Explique a grande rasgo como se realiza la codificación de variables numéricas a categóricas.

- Junte ambos dataset en uno, llamado **df_procesado**.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Separar variables numéricas y categóricas
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# --- Procesamiento de Variables Numéricas ---
# Crear un sub-DataFrame con solo las columnas numéricas
df_numerical = df[numerical_cols].copy()

# Rellenar datos vacíos con el promedio asociado a cada columna
for col in numerical_cols:
    if df_numerical[col].isnull().any():
        df_numerical[col] = df_numerical[col].fillna(df_numerical[col].mean())

# Normalizar los datos numéricos mediante MinMaxScaler
scaler = MinMaxScaler()
df_numerical_scaled = pd.DataFrame(scaler.fit_transform(df_numerical), columns=numerical_cols)

# --- Procesamiento de Variables Categóricas ---
# Crear un sub-DataFrame con solo las columnas categóricas
df_categorical = df[categorical_cols].copy()

# Aplicar one-hot encoding con get_dummies
df_categorical_encoded = pd.get_dummies(df_categorical, drop_first=False)

# --- Juntar ambos datasets ---
df_procesado = pd.concat([df_numerical_scaled, df_categorical_encoded], axis=1)

print("Dimensiones del DataFrame procesado:", df_procesado.shape)
display(df_procesado.head())

# Explicación de get_dummies
print("\nExplicación de la codificación de variables categóricas (get_dummies):")
print("El método `pd.get_dummies()` convierte variables categóricas en una representación numérica binaria (one-hot encoding).")
print("Para cada categoría única en una columna original, se crea una nueva columna.")
print("Si una fila pertenece a esa categoría, el valor en la nueva columna es 1; de lo contrario, es 0.")
print("Esto permite que los algoritmos de machine learning, que generalmente requieren entrada numérica, procesen variables categóricas.")

### 2.- Realizar ajuste mediante kmeans

Una vez depurado el conjunto de datos, es momento de aplicar el algoritmo de **kmeans**.

1. Ajuste el modelo de **kmeans** sobre el conjunto de datos, con un total de **8 clusters**.
2. Asociar a cada individuo el correspondiente cluster y calcular valor de los centroides de cada cluster.
3. Realizar un resumen de las principales cualidades de cada cluster. Para  esto debe calcular (para cluster) las siguientes medidas de resumen:
    * Valor promedio de las variables numérica
    * Moda para las variables numericas

In [ ]:
# 1. Ajustar el modelo de kmeans sobre el conjunto de datos, con un total de 8 clusters.
kmeans_model = KMeans(n_clusters=8, random_state=42, n_init=10)
kmeans_model.fit(df_procesado)

# 2. Asociar a cada individuo el correspondiente cluster
df_procesado['cluster'] = kmeans_model.labels_

# Calcular valor de los centroides de cada cluster.
centroids = kmeans_model.cluster_centers_
df_centroids = pd.DataFrame(centroids, columns=df_procesado.drop('cluster', axis=1).columns)

print("Dimensiones del DataFrame procesado con clusters:", df_procesado.shape)
print("Centroides de los clusters:")
display(df_centroids)

# 3. Realizar un resumen de las principales cualidades de cada cluster.
print("\nResumen de las principales cualidades de cada cluster:")

# Obtener los nombres de las columnas numéricas originales
original_numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

# Crear un DataFrame para almacenar los promedios y las modas
cluster_summary = pd.DataFrame()

for i in range(kmeans_model.n_clusters):
    cluster_data = df_procesado[df_procesado['cluster'] == i]

    # Calcular el promedio de las variables numéricas escaladas
    mean_values_scaled = cluster_data[original_numerical_cols].mean()

    # Para la moda, es más significativo usar los datos originales si se quiere interpretar categorías
    # Sin embargo, el ejercicio pide la moda para las variables NUMÉRICAS del df_procesado (escaladas).
    # Si se quisiera la moda de las categóricas originales, se debería trabajar con df_categorical
    mode_values_scaled = cluster_data[original_numerical_cols].mode().iloc[0] # .iloc[0] para obtener la primera moda si hay múltiples

    # Recopilar la información en un formato legible
    summary_row_mean = pd.Series(mean_values_scaled, name=f'Cluster {i} - Mean')
    summary_row_mode = pd.Series(mode_values_scaled, name=f'Cluster {i} - Mode')

    cluster_summary = pd.concat([cluster_summary, summary_row_mean.to_frame().T, summary_row_mode.to_frame().T])

display(cluster_summary)

### 3.- Elegir Número de cluster

Estime mediante la **regla del codo**, el número de cluster apropiados para el caso.
Para efectos prácticos, eliga la siguiente secuencia como número de clusters a comparar:

$$[5, 10, 20, 30, 50, 75, 100, 200, 300]$$

Una vez realizado el gráfico, saque sus propias conclusiones del caso.

In [ ]:
from sklearn.cluster import KMeans

wcss = [] # Suma de Cuadrados Intra-Cluster

# Define el rango de números de clusters a comparar
clusters_range = [5, 10, 20, 30, 50, 75, 100, 200, 300]

# Itera a través del rango de clusters definido
for num_clusters in clusters_range:
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    kmeans.fit(df_procesado)
    wcss.append(kmeans.inertia_)

# Grafica la curva del codo
plt.figure(figsize=(10, 6))
plt.plot(clusters_range, wcss, marker='o', linestyle='-')
plt.title('Método del Codo para Encontrar el Número Óptimo de Clusters')
plt.xlabel('Número de Clusters')
plt.ylabel('WCSS (Suma de Cuadrados Intra-Cluster)')
plt.xticks(clusters_range)
plt.grid(True)
plt.show()

Al observar el gráfico resultante, se pueden obtener conclusiones sobre el número apropiado de clusters. La regla del codo sugiere elegir el número de clusters donde la reducción en la inercia se estabiliza significativamente. En otras palabras, se busca el punto en el gráfico donde la curva de inercia comienza a aplanarse o forma un codo.

## Reducción de Dimensionalidad

<img src="https://1000logos.net/wp-content/uploads/2020/11/Wine-Logo-old.png" width = "300" align="center"/>


Para este ejercicio utilizaremos el **Wine Dataset**, un conjunto de datos clásico disponible en la librería **scikit-learn** y en el repositorio de la **UCI Machine Learning**.
Este dataset contiene información de **178 muestras de vino** provenientes de la región italiana de *Piamonte*. Cada vino pertenece a una de **tres variedades de uva** (*clases*), que actúan como etiquetas para el análisis supervisado, pero aquí se usarán solo como referencia en la visualización.

Cada muestra está descrita por **13 variables químicas** obtenidas de un análisis de laboratorio, entre ellas:

* **Alcohol**: porcentaje de alcohol en el vino.
* **Malic acid**: concentración de ácido málico.
* **Ash**: contenido de ceniza.
* **Alcalinity of ash**: alcalinidad de la ceniza.
* **Magnesium**: cantidad de magnesio (mg/L).
* **Total phenols**: concentración total de fenoles.
* **Flavanoids**: tipo de fenoles con propiedades antioxidantes.
* **Nonflavanoid phenols**: fenoles que no son flavonoides.
* **Proanthocyanins**: compuestos relacionados con el color y el sabor.
* **Color intensity**: intensidad del color del vino.
* **Hue**: matiz del color.
* **OD280/OD315 of diluted wines**: relación de absorbancia que mide la calidad del vino.
* **Proline**: concentración de prolina (un aminoácido).

Estas características permiten representar cada vino como un punto en un espacio de **13 dimensiones**.

El objetivo del análisis con este dataset es **reducir la dimensionalidad** para visualizar y explorar patrones en los datos. Para ello aplicaremos:

* **PCA (Principal Component Analysis):** identificar las combinaciones lineales de variables que explican la mayor varianza en el conjunto.
* **t-SNE (t-distributed Stochastic Neighbor Embedding):** mapear las muestras a 2D o 3D, preservando relaciones de vecindad y estructuras no lineales.

La comparación entre ambas técnicas permitirá observar cómo las tres clases de vinos se diferencian en el espacio reducido y discutir la utilidad de la reducción de dimensionalidad en datos con mayor número de variables que en el caso del dataset *Wine*.



In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

In [ ]:
# cargar dataset
dataset = load_wine()

# nombres de las variables
features = dataset.feature_names
target = 'wine_class'

# construir DataFrame
wine = pd.DataFrame(dataset.data, columns=features)
wine[target] = dataset.target

# ver primeras filas
wine.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,wine_class
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0




### 1. **Análisis detallado con PCA**

* Calcular la **varianza explicada** por cada componente principal y representar el gráfico de varianza acumulada, identificando cuántos componentes son necesarios para capturar al menos el **90–95% de la información**.
* Construir tablas y gráficos que muestren cómo las observaciones (vinos) se proyectan en las primeras componentes principales.
* Analizar los **loadings** (coeficientes de cada variable en los componentes) e interpretar qué características químicas del vino (alcohol, fenoles, color, etc.) tienen mayor influencia en las nuevas dimensiones.
* Visualizar los datos reducidos a 2D o 3D e interpretar si las **tres variedades de vino** se separan de forma clara en el espacio proyectado.



In [ ]:
from sklearn.preprocessing import StandardScaler

# Separar las características (X) de la variable objetivo (y)
X = wine.drop('wine_class', axis=1)
y = wine['wine_class']

# Escalar los datos antes de aplicar PCA
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X)

# Crear una instancia de PCA
pca = PCA()

# Ajustar PCA a los datos escalados
pca.fit(X_scaled)

# Calcular la varianza explicada por cada componente principal
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print("Varianza explicada por cada componente principal:", explained_variance_ratio)
print("Varianza acumulada explicada:", cumulative_explained_variance)

# Identificar cuántos componentes son necesarios para capturar al menos el 90-95% de la información.
# Encontrar el número de componentes para el 90%
num_components_90 = np.where(cumulative_explained_variance >= 0.90)[0][0] + 1
# Encontrar el número de componentes para el 95%
num_components_95 = np.where(cumulative_explained_variance >= 0.95)[0][0] + 1

print(f"\nSe necesitan {num_components_90} componentes para explicar al menos el 90% de la varianza.")
print(f"Se necesitan {num_components_95} componentes para explicar al menos el 95% de la varianza.")

# Gráfico de varianza explicada acumulada
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), cumulative_explained_variance, marker='o', linestyle='-')
plt.title('Varianza Acumulada Explicada por Componentes Principales')
plt.xlabel('Número de Componentes Principales')
plt.ylabel('Varianza Acumulada Explicada')
plt.axvline(x=num_components_90, color='r', linestyle='--', label=f'90% con {num_components_90} componentes')
plt.axvline(x=num_components_95, color='g', linestyle='--', label=f'95% con {num_components_95} componentes')
plt.grid(True)
plt.legend()
plt.show()

# Construir tablas y gráficos que muestren cómo las observaciones (vinos) se proyectan en las primeras componentes principales.

# Reducir a 2 componentes para visualización
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

# Crear un DataFrame para las componentes principales 2D
df_pca_2d = pd.DataFrame(data=X_pca_2d, columns=['PC1', 'PC2'])
df_pca_2d['wine_class'] = y

print("\nProyección de las observaciones en las primeras 2 Componentes Principales (PCA_2D):")
display(df_pca_2d.head())

# Visualizar los datos reducidos a 2D
plt.figure(figsize=(10, 8))
sns.scatterplot(x='PC1', y='PC2', hue='wine_class', data=df_pca_2d, palette='viridis', s=100, alpha=0.7)
plt.title('PCA 2D del Dataset Wine')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True)
plt.show()

# Reducir a 3 componentes para visualización
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

# Crear un DataFrame para las componentes principales 3D
df_pca_3d = pd.DataFrame(data=X_pca_3d, columns=['PC1', 'PC2', 'PC3'])
df_pca_3d['wine_class'] = y

print("\nProyección de las observaciones en las primeras 3 Componentes Principales (PCA_3D):")
display(df_pca_3d.head())

# Visualizar los datos reducidos a 3D
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(df_pca_3d['PC1'], df_pca_3d['PC2'], df_pca_3d['PC3'], c=df_pca_3d['wine_class'], cmap='viridis', s=100, alpha=0.7)

ax.set_title('PCA 3D del Dataset Wine')
ax.set_xlabel('Componente Principal 1')
ax.set_ylabel('Componente Principal 2')
ax.set_zlabel('Componente Principal 3')

legend1 = ax.legend(*scatter.legend_elements(), title='Wine Class')
ax.add_artist(legend1)

plt.show()

# Analizar los loadings (coeficientes de cada variable en los componentes)
# Extraer los loadings de las primeras componentes principales
loadings = pca.components_[:num_components_95] # Consideramos hasta el 95% de la varianza

df_loadings = pd.DataFrame(loadings.T, columns=[f'PC{i+1}' for i in range(loadings.shape[0])], index=X.columns)

print("\nLoadings de las variables en los componentes principales:")
display(df_loadings)

# Visualizar los loadings (por ejemplo, para PC1 y PC2)
plt.figure(figsize=(12, 6))
sns.heatmap(df_loadings.iloc[:, :3], cmap='viridis', annot=True, fmt=".2f")
plt.title('Loadings de las variables en los primeros 3 Componentes Principales')
plt.xlabel('Componentes Principales')
plt.ylabel('Variables Originales')
plt.show()



### 2. **Análisis detallado con t-SNE**

* Aplicar **t-SNE** para reducir los datos a 2 dimensiones, probando diferentes configuraciones de hiperparámetros como *perplexity* y *learning rate*.
* Comparar las distintas visualizaciones obtenidas y discutir cómo los hiperparámetros afectan la estructura de los clústeres.
* Analizar si las **tres clases de vinos** forman agrupaciones definidas y si t-SNE logra capturar relaciones no lineales que PCA no refleja.



In [ ]:
# Función para graficar t-SNE
def plot_tsne(X_tsne, y, perplexity, learning_rate, n_iter, title):
    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x=X_tsne[:, 0],
        y=X_tsne[:, 1],
        hue=y,
        palette='viridis',
        legend='full',
        alpha=0.7,
        s=100
    )
    plt.title(f't-SNE 2D del Dataset Wine (Perplexity={perplexity}, LR={learning_rate}, Iter={n_iter})\n{title}')
    plt.xlabel('Componente t-SNE 1')
    plt.ylabel('Componente t-SNE 2')
    plt.grid(True)
    plt.show()

# --- t-SNE con diferentes hiperparámetros ---

# Ejemplo 1: Perplexity estándar, Learning Rate estándar
print("\n--- Ejecutando t-SNE: Perplexity=30, Learning Rate=200, n_iter=1000 ---")
tsne1 = TSNE(n_components=2, perplexity=30, learning_rate=200, n_iter=1000, random_state=42)
X_tsne1 = tsne1.fit_transform(X_scaled)
plot_tsne(X_tsne1, y, perplexity=30, learning_rate=200, n_iter=1000, title='Configuración Estándar')

# Ejemplo 2: Perplexity baja
print("\n--- Ejecutando t-SNE: Perplexity=5, Learning Rate=200, n_iter=1000 ---")
tsne2 = TSNE(n_components=2, perplexity=5, learning_rate=200, n_iter=1000, random_state=42)
X_tsne2 = tsne2.fit_transform(X_scaled)
plot_tsne(X_tsne2, y, perplexity=5, learning_rate=200, n_iter=1000, title='Perplexity Baja: Enfatiza Vecinos Cercanos')

# Ejemplo 3: Perplexity alta
print("\n--- Ejecutando t-SNE: Perplexity=50, Learning Rate=200, n_iter=1000 ---")
tsne3 = TSNE(n_components=2, perplexity=50, learning_rate=200, n_iter=1000, random_state=42)
X_tsne3 = tsne3.fit_transform(X_scaled)
plot_tsne(X_tsne3, y, perplexity=50, learning_rate=200, n_iter=1000, title='Perplexity Alta: Considera Vecinos Lejanos')

# Ejemplo 4: Learning Rate bajo
print("\n--- Ejecutando t-SNE: Perplexity=30, Learning Rate=10, n_iter=1000 ---")
tsne4 = TSNE(n_components=2, perplexity=30, learning_rate=10, n_iter=1000, random_state=42)
X_tsne4 = tsne4.fit_transform(X_scaled)
plot_tsne(X_tsne4, y, perplexity=30, learning_rate=10, n_iter=1000, title='Learning Rate Bajo: Convergencia Lenta')

# Ejemplo 5: Learning Rate alto
print("\n--- Ejecutando t-SNE: Perplexity=30, Learning Rate=1000, n_iter=1000 ---")
tsne5 = TSNE(n_components=2, perplexity=30, learning_rate=1000, n_iter=1000, random_state=42)
X_tsne5 = tsne5.fit_transform(X_scaled)
plot_tsne(X_tsne5, y, perplexity=30, learning_rate=1000, n_iter=1000, title='Learning Rate Alto: Riesgo de Divergencia')

print("\n--- Conclusiones sobre los hiperparámetros de t-SNE ---")
print("**Perplexity:** Controla el balance entre considerar vecinos cercanos y lejanos. Valores bajos se enfocan en la estructura local, mientras que valores altos buscan la estructura global. Un valor típico está entre 5 y 50.")
print("**Learning Rate:** Determina el tamaño del paso en la optimización. Un valor muy bajo puede hacer que la convergencia sea muy lenta, mientras que un valor muy alto puede llevar a una mala optimización y a una disposición de puntos inestable.")
print("**n_iter:** El número de iteraciones. Más iteraciones pueden mejorar la calidad del embedding, pero aumentan el tiempo de cómputo.")
print("En general, es crucial experimentar con estos hiperparámetros para encontrar la visualización que mejor represente la estructura intrínseca de los datos.")



### 3. **Comparación entre PCA y t-SNE**

* Contrastar las visualizaciones y discutir las **ventajas y limitaciones** de cada técnica:

  * PCA como método **lineal** para interpretar varianza y relaciones globales.
  * t-SNE como método **no lineal** que preserva relaciones locales y vecindades.
* Evaluar en qué escenarios prácticos sería más recomendable usar PCA (interpretabilidad, reducción previa para modelos) o t-SNE (exploración y visualización de clústeres).
* Reflexionar sobre la **importancia de la reducción de dimensionalidad** en datasets de alta dimensión como Wine, destacando su utilidad para:

  * Visualizar patrones ocultos en los datos.
  * Reducir complejidad y ruido antes de aplicar algoritmos de aprendizaje automático.
  * Facilitar la interpretación y comunicación de resultados.



In [ ]:
### Comparación Final: PCA vs. t-SNE en el Dataset Wine

**1. Calidad de la Visualización:**
* **PCA:** Muestra una separación lineal. Aunque identifica grupos, hay solapamiento entre las clases 1 y 2. Es mejor para ver la estructura global y la dispersión.
* **t-SNE:** Logra una separación mucho más nítida y compacta. Es superior para identificar clústeres definidos mediante relaciones no lineales.

**2. Cuándo usar cada uno:**
* **PCA:** Úsalo cuando necesites velocidad, cuando quieras saber exactamente qué variables (como el alcohol o los fenoles) afectan más a la varianza (vía *loadings*), o como paso previo para otros modelos.
* **t-SNE:** Úsalo exclusivamente para exploración visual y para encontrar estructuras locales ocultas que los métodos lineales no ven.

**3. Conclusión sobre Dimensionalidad:**
Reducir de 13 variables a 2 o 3 nos permite 'ver' la química del vino. Sin estas técnicas, sería imposible para el ojo humano detectar que los vinos se agrupan tan claramente según su composición química. El laboratorio queda concluido exitosamente con estas observaciones.